# Package Download

In [1]:
!python -m spacy download de_core_news_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.6/14.6 MB 80.4 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('de_core_news_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


# Restart Enviroment!

In [1]:

# Necessary for Colab
#Package contains PDF library
!pip install -U chromadb pypdf langchain-community sentence-transformers


import requests
from io import BytesIO
from langchain_community.document_loaders import PyPDFLoader



     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 45.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 338.8/338.8 kB 32.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 59.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 588.7/588.7 kB 45.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 29.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 98.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 67.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 548.1/548.1 kB 38.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 86.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.1/72.1 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.2/180.2 kB 18.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.0

#I Text Chunking

In [2]:
from langchain_community.document_loaders import PyPDFDirectoryLoader

DATA_PATH = r"sample_data"
loader = PyPDFDirectoryLoader(DATA_PATH)

documents_from_url = loader.load()


#I-3 SpacyTextSplitter

In [3]:
from langchain_text_splitters import SpacyTextSplitter
import spacy.cli
from langchain_core.documents import Document
from datetime import datetime

spacy.load('de_core_news_sm')

# be careful to use the right dictionary, see: https://spacy.io/models
text_splitter = SpacyTextSplitter(
    pipeline="de_core_news_sm",
    chunk_size=500,
    chunk_overlap=50
    )

rawtext = text_splitter.split_documents(documents_from_url)

chunks = []

for i, doc_chunk in enumerate(rawtext):
    # Create a new Document object with added metadata
    new_metadata = {
        "chunk_id": i,
        "source": "",
        "orig_source": "AnySource",
        "mandant": "OtherCompany",
        "last_updated": datetime.now().isoformat(),
        "language": "de",
        **doc_chunk.metadata # Include existing metadata from the chunk
    }
    doc = Document(
        page_content=doc_chunk.page_content, # Correctly access content from the current Document object
        metadata=new_metadata
    )
    chunks.append(doc)


i = 0

for chunk in chunks:
    print(chunk)
    print('####################################################################')

    i += 1

page_content='Basisnormen für Tischler und Schreiner 
Fachregeln 


Normen gelten nicht selten als „allgemein anerkannte 
Regel der Technik (aaRdT)".

Daher ist es notwendig, 
jene Normen zu kennen, die für die jeweiligen Auf-
träge relevant sind. 


Mit dieser kleinen Zusammenstellung soll ein grober 
Überblick zu wichtigen Normen für Tischler und 
Schreiner gegeben werden.

Da Normen dem Copy-
right unterliegen, ist ein Abdruck von Normen an die-
ser Stelle nicht möglich.' metadata={'chunk_id': 0, 'source': 'sample_data/13_Basisnormen-für-Tischler-und-Schreiner.pdf', 'orig_source': 'AnySource', 'mandant': 'OtherCompany', 'last_updated': '2026-05-14T14:04:59.172218', 'language': 'de', 'producer': 'Scanner System Image Conversion', 'creator': 'Scanner System', 'creationdate': '2019-08-20T11:09:22+01:00', 'moddate': '2019-08-20T11:09:22+01:00', 'total_pages': 12, 'page': 0, 'page_label': '1'}
####################################################################
page_content='Normen könne

#III-1 Create Embeddings using Qwen3.5 0.6b and Insert into Vector Database

In this example, embeddings are created using a custom model. Also an example with an built-in model is shown.

In [5]:
import chromadb
from chromadb.utils.embedding_functions import OllamaEmbeddingFunction, SentenceTransformerEmbeddingFunction

# setting the environment
# By default, Chromadb uses all-MiniLM-L6-v2 do create embeddings

CHROMA_PATH = r"chroma_db"

chroma_client = chromadb.PersistentClient(path=CHROMA_PATH)

# 1. Define the Hugging Face / Sentence Transformers Embedding Function
# This points directly to Alibaba's official model repository on Hugging Face
qwen_ef = SentenceTransformerEmbeddingFunction(
    model_name="Qwen/Qwen3-Embedding-0.6B"
)

# 2. Pass the embedding function to the collection
# TIP: Use a new collection name if you previously used a different embedding model!
collection = chroma_client.get_or_create_collection(
    name="Furniture_Qwen",
    embedding_function=qwen_ef
)


# preparing to be added in chromadb

documents = []
metadata = []
ids = []

i = 0

for chunk in chunks:
    documents.append(chunk.page_content)
    ids.append("ID"+str(i))
    metadata.append(chunk.metadata)
    print(chunk.page_content)
    print('############################')

    i += 1

# adding to chromadb


collection.upsert(
    documents=documents,
    metadatas=metadata,
    ids=ids
)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/215 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.19G [00:00<?, ?B/s]

...

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/313 [00:00<?, ?B/s]

Basisnormen für Tischler und Schreiner 
Fachregeln 


Normen gelten nicht selten als „allgemein anerkannte 
Regel der Technik (aaRdT)".

Daher ist es notwendig, 
jene Normen zu kennen, die für die jeweiligen Auf-
träge relevant sind. 


Mit dieser kleinen Zusammenstellung soll ein grober 
Überblick zu wichtigen Normen für Tischler und 
Schreiner gegeben werden.

Da Normen dem Copy-
right unterliegen, ist ein Abdruck von Normen an die-
ser Stelle nicht möglich.
############################
Normen können über 
www.beuth.de  bezogen werden.

Sinnvoll ist dabei 
meist die Anschaffung von DIN-Taschenbüchern, in 
welchen Normen nach Themen zusammengestellt 
sind.

So gibt es z. B. das „Normenhandbuch Tischler-
und Schreinerhandwerk" in welchem viele Normen 
abgebildet sind.

In den DIN-Taschenbüchern 359 und 
360 finden sich z. B. Nomen für die Holzwirtschaft.
############################
Wer viel mit Normen arbeiten muss, kann sich auch 
bestimmter DIN-Portale bedienen, auf welchen man 
zah

In [6]:
while True:

  user_query = input("How can I help you?")

  if user_query.lower() == "quit":
            break

  results = collection.query(
      query_texts=[user_query],
      n_results=10

  )

  print(results['documents'])
  print(results['metadatas'])



How can I help you?Welche Regeln gibt es bei TIschlern?
[['Basisnormen für Tischler und Schreiner \nFachregeln \n\n\nNormen gelten nicht selten als „allgemein anerkannte \nRegel der Technik (aaRdT)".\n\nDaher ist es notwendig, \njene Normen zu kennen, die für die jeweiligen Auf-\nträge relevant sind. \n\n\nMit dieser kleinen Zusammenstellung soll ein grober \nÜberblick zu wichtigen Normen für Tischler und \nSchreiner gegeben werden.\n\nDa Normen dem Copy-\nright unterliegen, ist ein Abdruck von Normen an die-\nser Stelle nicht möglich.', 'Sie werden allerdings ggf. zitiert, wenn es zu \nSchäden/Reklamationen kommt. \n\n\nFür Tischler sind innerhalb des Taschenbuches insbe-\nsondere Normen zu folgenden Bereichen von Bedeu-\ntung: \nBetten \n- Etagenbetten (Hochbetten) \n- Krippen und Wiegen \n\n\nIn ähnlicher Weise existiert noch ein DIN-Taschenbuch \nNr. 467 für Büro-, Schul- und Objektmöbel. \n\n\nDie vorstehenden Aussagen treffen vergleichbar zu.', 'Die Suchergebnisse sind auf Normen

KeyboardInterrupt: Interrupted by user